# 🧠 Notebook 05 — CNN Baseline Training (V2 - Optimized)

---

## Project Context

| Field | Detail |
|---|---|
| **Competition** | SATRIA DATA 2026 — Big Data Challenge |
| **Experiment** | Baseline CNN — Experiment 01 |
| **Stage** | Modeling (Step 3 of 7) |
| **Primary Metric** | Macro-averaged F1 Score |

---

## Arsitektur & Hyperparameter

Berdasarkan `configs/baseline.yaml`, kita akan melatih model **Custom CNN** dengan 4 Convolutional Block.
- **Optimizer**: AdamW (LR=0.001)
- **Scheduler**: ReduceLROnPlateau
- **Loss Function**: CrossEntropyLoss
- **Early Stopping**: Berdasarkan `val_macro_f1`

**Catatan:** Notebook ini menggunakan modul `Trainer` dari `src/training/trainer.py` untuk menjaga *Clean Architecture*.

In [ ]:
# ============================================================
# CELL 1 — IMPORTS & SETUP
# ============================================================
import sys
import logging
from pathlib import Path
import yaml

PROJECT_ROOT = Path('..').resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.utils.seed import set_seed
from src.preprocessing.dataloader import build_dataloaders
from src.models.cnn.baseline import build_baseline_cnn
from src.training.trainer import Trainer

logging.basicConfig(level=logging.INFO, format='%(asctime)s | %(message)s', datefmt='%H:%M:%S')
logger = logging.getLogger(__name__)

with open(PROJECT_ROOT / 'configs' / 'baseline.yaml', 'r') as f:
    config = yaml.safe_load(f)

SEED = config['experiment']['seed']
set_seed(SEED)

---
## Step 1 — Siapkan DataLoaders
Kita membangun ulang loader di sini dengan `batch_size=32`.

In [7]:
# ============================================================
# CELL 2 — DATALOADERS (Load from Preprocessing Step)
# ============================================================
import pandas as pd

reports_dir = PROJECT_ROOT / config['output']['reports_dir']
csv_path = reports_dir / 'preprocessing_split.csv'

logger.info(f"Loading dataset split dari {csv_path}")
df_all = pd.read_csv(csv_path)

df_train = df_all[df_all['split'] == 'train'].reset_index(drop=True)
df_val = df_all[df_all['split'] == 'val'].reset_index(drop=True)

train_loader, val_loader = build_dataloaders(
    df_train=df_train,
    df_val=df_val,
    config=config,
    use_weighted_sampler=True
)

11:39:35 | Loading dataset split dari D:\Data Analysis\Smart Waste Classification\outputs\reports\preprocessing_split.csv
11:39:35 | WasteDataset [train] dibuat: 21,219 gambar, transform=Yes
11:39:35 | WeightedRandomSampler diaktifkan untuk training.
11:39:35 | Train DataLoader: 21,219 gambar, batch_size=32, num_batches=663
11:39:35 | WasteDataset [val] dibuat: 5,305 gambar, transform=Yes
11:39:35 | Val DataLoader: 5,305 gambar, batch_size=32, num_batches=166


---
## Step 2 — Inisialisasi Model
Membangun `BaselineCNN` dari scratch tanpa pre-trained weights.

In [8]:
# ============================================================
# CELL 3 — MODEL INITIALIZATION
# ============================================================
num_classes = len(config['data']['class_names'])
model = build_baseline_cnn(config=config)

print(f"Model Architecture:\n{model}")

Model Architecture:
BaselineCNN(
  conv_channels  = [32, 64, 128, 256]
  num_classes    = 3
  total_params   = 422,179
  trainable      = 422,179
)


---
## Step 3 — Training Loop
Memanggil class `Trainer` yang membungkus AdamW, ReduceLROnPlateau, EarlyStopping, dan logging.

In [10]:
# Force reload module trainer.py agar mengambil versi terbaru dari disk
import importlib
import src.training.trainer
importlib.reload(src.training.trainer)
from src.training.trainer import Trainer

# ============================================================
# CELL 4 — TRAIN
# ============================================================
trainer = Trainer(
    model=model,
    config=config,
    project_root=PROJECT_ROOT
)

# Training akan mencetak metrik per epoch
history = trainer.train(train_loader, val_loader)
logger.info("Proses training selesai!")

11:41:52 | ============================================================
11:41:52 |   TRAINING DIMULAI: baseline_cnn_exp01
11:41:52 |   Device   : cpu
11:41:52 |   Epochs   : 30
11:41:52 |   Optimizer: AdamW (lr=0.001)
11:41:52 |   Scheduler: ReduceLROnPlateau
11:41:52 |   Early Stop: patience=5
11:41:52 | ============================================================
Train Epoch 1:   0%|          | 0/663 [00:00<?, ?it/s]c:\Users\Microsoft\miniconda3\Lib\site-packages\torch\utils\data\dataloader.py:752: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
Val Epoch 1:   0%|          | 0/166 [00:00<?, ?it/s]                         c:\Users\Microsoft\miniconda3\Lib\site-packages\torch\utils\data\dataloader.py:752: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
11:58:37 | Epoch [01/30] | train_loss=0

KeyboardInterrupt: 